# Forecasting BC Hydro load for 1-7 January 2026 with MOIRAI

**Goal:** use BC Hydro's hourly load from 2016-2025 to forecast the **first week of January 2026**
(168 hours) with **MOIRAI 2.0** (Salesforce's pretrained forecasting model), then compare the
forecast with the actual 2026 load.

MOIRAI is used **zero-shot**: we don't train it. It reads the history and continues the pattern.

**How to run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

## 1. Install MOIRAI

In [ ]:
!pip install -q --no-deps uni2ts==2.0.0
!pip install -q gluonts einops jaxtyping hydra-core lightning xlrd

## 2. Connect Google Drive and list the data files

In [ ]:
import glob, os
from google.colab import drive
drive.mount("/content/drive")

DATA_FOLDER = "/content/drive/MyDrive/Bchydrodata/Input"   # folder with the BalancingAuthorityLoad files

files = sorted(glob.glob(os.path.join(DATA_FOLDER, "*.xls*")))
for f in files:
    print(os.path.basename(f))

## 3. Read all files into one hourly series

Each file has a few title rows, then three columns: **Date**, **HE** (hour ending, 1-24) and **Load** (MW).
Two daylight-saving quirks are cleaned up so there is exactly one value per hour:
* in March the skipped hour shows as `0` or blank: removed and filled in from its neighbours;
* in November the repeated hour is marked `2*`: removed.

In [ ]:
import pandas as pd

def read_file(path):
    try:
        raw = pd.read_excel(path, header=None)
    except Exception:                                   # some BC Hydro .xls files are really HTML tables
        raw = pd.DataFrame(pd.read_html(path)[0].values)
    he = raw.iloc[:, 1].astype(str).str.strip().str.upper()
    header_row = he[he == "HE"].index[0]                # the row that says Date | HE | Load
    df = raw.iloc[header_row + 1:, :3]
    df.columns = ["Date", "HE", "Load"]
    return df

df = pd.concat([read_file(f) for f in files], ignore_index=True)

df = df[~df["HE"].astype(str).str.contains("*", regex=False)]           # drop November repeated hour
df["Load"] = pd.to_numeric(df["Load"], errors="coerce")
df = df[df["Load"] > 0]                                                   # drop March 0 / blank hour
df["Time"] = (pd.to_datetime(df["Date"], format="mixed")
              + pd.to_timedelta(pd.to_numeric(df["HE"]) - 1, unit="h"))  # HE 1 = 00:00-01:00

load = df.groupby("Time")["Load"].mean().sort_index()
load = load.asfreq("h").interpolate()                                    # one value every hour

print(f"{len(load):,} hours from {load.index[0]} to {load.index[-1]}")

## 4. Split: history (2016-2025) and the week to forecast (1-7 Jan 2026)

In [ ]:
history = load[:"2025-12-31 23:00"]
actual = load["2026-01-01 00:00":"2026-01-07 23:00"]

print(f"History: {history.index[0]} to {history.index[-1]}  ({len(history):,} hours, {history.index.year.nunique()} years)")
print(f"Actual : {actual.index[0]} to {actual.index[-1]}  ({len(actual)} hours)")
assert len(actual) == 168, "The 2026 file must cover 1-7 January"

In [ ]:
import matplotlib.pyplot as plt

daily = history.resample("D").mean()
plt.figure(figsize=(14, 3.5))
plt.plot(daily.index, daily.values, color="#52514e", linewidth=0.7)
plt.title("BC Hydro daily average load, 2016-2025 (the history MOIRAI learns from)", loc="left")
plt.ylabel("MW"); plt.grid(alpha=0.3)
os.makedirs("results", exist_ok=True)
plt.savefig("results/history.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Forecast 1-7 January 2026 with MOIRAI

MOIRAI reads the most recent `CONTEXT_HOURS` of history (it has a maximum input length, so it can't
read all 10 years at once; 8,064 hours is the last 48 weeks of 2025) and forecasts the next 168 hours.

In [ ]:
import numpy as np
import torch
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

CONTEXT_HOURS = 8064    # how much recent history MOIRAI looks at
HORIZON = 168           # 7 days x 24 hours

model = Moirai2Forecast(
    module=Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small"),
    prediction_length=HORIZON, context_length=CONTEXT_HOURS,
    target_dim=1, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
)
model.to("cuda" if torch.cuda.is_available() else "cpu").eval()

with torch.no_grad():
    out = model.predict([history.values[-CONTEXT_HOURS:].astype("float32")])   # (1, 9 quantiles, 168)

q = list(model.module.quantile_levels)          # 0.1, 0.2, ... 0.9
result = pd.DataFrame({
    "Actual": actual.values,
    "MOIRAI": out[0, q.index(0.5)],             # median = best guess
    "MOIRAI_low": out[0, q.index(0.1)],         # 80% of actual values should fall
    "MOIRAI_high": out[0, q.index(0.9)],        # between low and high
}, index=actual.index)
result.head()

## 6. A simple baseline to compare against

**Same week last year:** the load from the matching days in early January 2025 (364 days earlier,
so Mondays line up with Mondays). If MOIRAI can't beat this, it isn't adding much.

In [ ]:
result["Last_year"] = history.reindex(actual.index - pd.Timedelta(days=364)).values
result.head()

## 7. Forecast vs actual

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(result.index, result["Actual"], color="#52514e", linewidth=2.2, label="Actual 2026")
plt.fill_between(result.index, result["MOIRAI_low"], result["MOIRAI_high"],
                 color="#eb6834", alpha=0.15, label="MOIRAI 80% range")
plt.plot(result.index, result["MOIRAI"], color="#eb6834", linewidth=1.8, label="MOIRAI forecast")
plt.plot(result.index, result["Last_year"], color="#2a78d6", linewidth=1.2, linestyle="--",
         label="Same week last year")
plt.title("BC Hydro hourly load, 1-7 January 2026", loc="left")
plt.ylabel("MW"); plt.grid(alpha=0.3); plt.legend(loc="upper left", ncol=2)
plt.savefig("results/forecast_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Accuracy

* **MAE** (MW): average size of the hourly error.
* **RMSE** (MW): like MAE but big misses count more.
* **MAPE** (%): average error as a percentage of the actual load.
* **Max error** (MW): the single worst hour.

In [ ]:
def score(forecast):
    err = result["Actual"] - forecast
    return {"MAE (MW)": err.abs().mean(),
            "RMSE (MW)": np.sqrt((err ** 2).mean()),
            "MAPE (%)": (err.abs() / result["Actual"]).mean() * 100,
            "Max error (MW)": err.abs().max()}

scores = pd.DataFrame({"MOIRAI": score(result["MOIRAI"]),
                       "Same week last year": score(result["Last_year"])}).T.round(2)

inside = ((result["Actual"] >= result["MOIRAI_low"]) & (result["Actual"] <= result["MOIRAI_high"])).mean() * 100
print(f"Actual load fell inside MOIRAI's 80% range in {inside:.0f}% of hours\n")
scores

### Day by day

In [ ]:
by_day = result.resample("D").mean()[["Actual", "MOIRAI", "Last_year"]]
by_day["MOIRAI error (%)"] = (by_day["MOIRAI"] - by_day["Actual"]) / by_day["Actual"] * 100
by_day.index = by_day.index.strftime("%a %d %b")
by_day.round(1)

## 9. Save the results

Everything goes into a `results` folder (2 graphs + 3 tables), zipped as `results.zip` and downloaded.
Unzip it and upload the files into the `MOIRAI/results/` folder on GitHub so others can see them.

In [ ]:
import shutil
result.round(1).to_csv("results/hourly_forecast.csv")
scores.to_csv("results/scores.csv")
by_day.round(1).to_csv("results/daily_comparison.csv")
shutil.make_archive("results", "zip", "results")
from google.colab import files
files.download("results.zip")

## Things to mention in the report

* **New Year's Day (1 Jan) is a holiday.** MOIRAI only sees load numbers, not the calendar, so it
  doesn't know 1 Jan is special. Check the day-by-day table: that day usually has the biggest error.
* **No weather input.** The lecture notes list temperature as the main driver of load. A cold snap
  in early January that wasn't in the history can't be predicted from load alone.
* **Try it yourself:** change `CONTEXT_HOURS` in step 5 (e.g. `2016` = 12 weeks, `4032` = 24 weeks)
  and re-run steps 5 to 8 to see whether more history helps.